# Search Parameter Tuning Manually (we could use hyperopt, grid search, or random search)

source: https://www.youtube.com/watch?v=rSBSS_kCYN0&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv

Note: we need to do for vector and hyber Search also

### Parameters Tuning

In [1]:
import sys
import os
import json
import pandas as pd
from rich import print
from minsearch import Index


parent_directory = os.path.dirname(os.getcwd())
sys.path.append(parent_directory)
from evaluation_utils import compute_relevance_total, hit_rate, mean_reciprocal_rank
from ingest import (load_faq_data,
                    build_index,
                    text_search
) 

# load the documents, fillter llm_zoocamp ONLY since the ground_truth data is generated from the llm_zoomcamp ONLY for simplicity
documents = load_faq_data()
documents_llm = [doc for doc in documents if doc['course'] == "llm-zoomcamp"]
print(f'The number of documents = {len(documents_llm)}')

# get the created ground-truth data
df_ground_truth = pd.read_csv('./data/ground_truth_FAQ_data.csv')
ground_truth = df_ground_truth.to_dict(orient="records")
# convert the df to dictionary

# note some answers do not have 5 questions
print(f'The number of ground_truth = {len(ground_truth)}')


# create search index and fit with document
search_index = build_index(documents_llm)

# Bind your fitted index to the search function using a lambda
bound_text_search = lambda query: text_search(query=query, search_index=search_index)

The number of documents = 103

The number of ground_truth = 510

In [2]:
def evaluate(ground_truth, search_function):
    relavent_matrix  = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relavent_matrix ),
        "mean_reciprocal_rank": mean_reciprocal_rank(relavent_matrix ),
    }

# get the created ground-truth data
df_ground_truth = pd.read_csv('./data/ground_truth_FAQ_data.csv')
ground_truth = df_ground_truth.to_dict(orient="records")


# create search index and fit with document
search_index = build_index(documents_llm)
# 1. Bind your fitted index to the search function using a lambda
bound_text_search = lambda query: text_search(query=query, search_index=search_index)

evaluate(ground_truth, bound_text_search)

{'hit_rate': 0.7509803921568627, 'mean_reciprocal_rank': 0.6142156862745095}

# Parameter Tuning with evaluate

In [3]:
def text_search_tuning(query, search_index, question_boost, section_boost,answer_boost):
    """
        search_id is the minsearch index that is fitted with the documents
        we use it for building ground truth, then we emphasize the question
    """
    boost_dict = {"question": question_boost,
                  "section": section_boost,
                   "answer": answer_boost,
                  }
    return search_index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

# use text_search_tuning with evaluate, 
# start with question_boost
question_boosts = [0.5, 1.0, 3.0, 5.0, 10.0]



for question_boost in question_boosts:
    bound_text_search_tuning = lambda query: text_search_tuning(query=query,
                                                         search_index=search_index,
                                                         question_boost=question_boost,
                                                         section_boost=1,
                                                         answer_boost=1)
    print(f'For question_boost = {question_boost}: We get {evaluate(ground_truth, bound_text_search_tuning)}')

    # so from the results, we see thay question_boost = 1 is the best, so we need to boost answer instead of question  

For question_boost = 0.5: We get {'hit_rate': 0.8058823529411765, 'mean_reciprocal_rank': 0.708954248366013}

For question_boost = 1.0: We get {'hit_rate': 0.8235294117647058, 'mean_reciprocal_rank': 0.6936928104575161}

For question_boost = 3.0: We get {'hit_rate': 0.7372549019607844, 'mean_reciprocal_rank': 0.604052287581699}

For question_boost = 5.0: We get {'hit_rate': 0.711764705882353, 'mean_reciprocal_rank': 0.5868627450980392}

For question_boost = 10.0: We get {'hit_rate': 0.6843137254901961, 'mean_reciprocal_rank': 0.5535294117647058}

# Tuning all parameters manually

In [5]:
# use text_search_tuning with evaluate, 
# start with question_boost
question_boosts = [0.5, 1.0,2.0, 3.0, 5.0, 10.0]
answer_boosts = [0.5,1.0, 2.0, 4.0, 10.0]
section_boosts = [0.5, 0.1, 0.2, 0.5]
from tqdm import tqdm
evaluation_results = list()


for question_boost in tqdm(question_boosts, desc="Tuning parameters"):
    for answer_boost in tqdm(answer_boosts):
        for section_boost in tqdm(section_boosts):

            bound_text_search_tuning = lambda query: text_search_tuning(
                                                query=query,
                                                search_index=search_index,
                                                question_boost=question_boost,
                                                section_boost=section_boost,
                                                answer_boost=answer_boost)
            
            metrics = evaluate(ground_truth, bound_text_search_tuning)
            evaluation_results.append({
                "section_boost": section_boost,
                "answer_boost":answer_boost,
                "question_boost":question_boost,
                "hit_rate":metrics["hit_rate"],
                "mean_reciprocal_rank":metrics["mean_reciprocal_rank"]
            })



Tuning parameters:   0%|          | 0/6 [00:00<?, ?it/s]




100%|██████████| 4/4 [00:02<00:00,  1.47it/s]





100%|██████████| 4/4 [00:02<00:00,  1.45it/s]





100%|██████████| 4/4 [00:02<00:00,  1.43it/s]





100%|██████████| 4/4 [00:02<00:00,  1.48it/s]





Tuning parameters:  17%|█▋        | 1/6 [00:13<01:08, 13.75s/it]




100%|██████████| 4/4 [00:02<00:00,  1.49it/s]





100%|██████████| 4/4 [00:02<00:00,  1.47it/s]





100%|██████████| 4/4 [00:02<00:00,  1.39it/s]





100%|██████████| 4/4 [00:02<00:00,  1.41it/s]





Tuning parameters:  33%|███▎      | 2/6 [00:27<00:55, 13.98s/it]




100%|██████████| 4/4 [00:02<00:00,  1.48it/s]





100%|██████████| 4/4 [00:02<00:00,  1.47it/s]





100%|██████████| 4/4 [00:02<00:00,  1.43it/s]





100%|██████████| 4/4 [00:02<00:00,  1.49it/s]





Tuning parameters:  50%|█████     | 3/6 [00:41<00:41, 13.82s/it]




100%|██████████| 4/4 [00:02<00:00,  1.48it/s]





100%|██████████| 4/4 [00:02<00:00,  1.48it/s]





100%|██████████| 4

In [6]:
df_evaluation_results = pd.DataFrame(evaluation_results)
df_evaluation_results.sort_values(by="mean_reciprocal_rank", ascending=False).head(10)

,section_boost,answer_boost,question_boost,hit_rate,mean_reciprocal_rank
79,0.5,10.0,3.0,0.941176,0.845719
76,0.5,10.0,3.0,0.941176,0.845719
78,0.2,10.0,3.0,0.941176,0.843105
9,0.1,2.0,0.5,0.937255,0.841732
34,0.2,4.0,1.0,0.937255,0.841732
77,0.1,10.0,3.0,0.943137,0.840948
33,0.1,4.0,1.0,0.937255,0.840654
10,0.2,2.0,0.5,0.931373,0.839412
59,0.5,10.0,2.0,0.935294,0.839085
56,0.5,10.0,2.0,0.935294,0.839085


In [ ]:
# text_search_tuning with optimal parameters
def text_search_tuning(query, search_index, question_boost=3.0, section_boost=0.5,answer_boost=10):
    """
        search_id is the minsearch index that is fitted with the documents
        we use it for building ground truth, then we emphasize the question
    """
    boost_dict = {"question": question_boost,
                  "section": section_boost,
                   "answer": answer_boost,
                  }
    return search_index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )


